# Transformer (Attention-Based) - TensorFlow / Keras


In [1]:
"""
Transformer Model for Text Classification using TensorFlow/Keras
----------------------------------------------------------------

Task:
    Sentiment Classification on IMDB Movie Reviews

Dataset:
    TensorFlow Datasets (tfds) - IMDB Reviews

Features:
    - Clean and modular code
    - Transformer Encoder implementation
    - Text preprocessing with TextVectorization
    - Training + Evaluation
    - Easy to adapt to other NLP datasets

TensorFlow Version:
    >= 2.10 recommended
"""

import tensorflow as tf
from tensorflow.keras import layers
import tensorflow_datasets as tfds
import numpy as np

# ============================================================
# 1. CONFIGURATION
# ============================================================

VOCAB_SIZE = 20000
MAX_LEN = 200
EMBED_DIM = 128
NUM_HEADS = 4
FF_DIM = 128
BATCH_SIZE = 32
EPOCHS = 3

AUTOTUNE = tf.data.AUTOTUNE


# ============================================================
# 2. LOAD DATASET
# ============================================================

print("Loading IMDB dataset...")

(train_ds, test_ds), ds_info = tfds.load("imdb_reviews",
                                         split=["train", "test"],
                                         as_supervised=True,
                                         with_info=True)

# Split training into train + validation
train_size = 20000
val_size = 5000

train_data = train_ds.take(train_size)
val_data = train_ds.skip(train_size).take(val_size)

print("Dataset loaded successfully!")


# ============================================================
# 3. TEXT PREPROCESSING
# ============================================================

# Text vectorization layer converts text -> integer tokens
vectorize_layer = layers.TextVectorization(max_tokens=VOCAB_SIZE,
                                           output_mode="int",
                                           output_sequence_length=MAX_LEN)

# Adapt vocabulary from training text
train_text = train_data.map(lambda text, label: text)
vectorize_layer.adapt(train_text)

print("Vocabulary size:", len(vectorize_layer.get_vocabulary()))


def preprocess(text, label):
    """
    Converts raw text into token IDs.
    """
    text = vectorize_layer(text)
    return text, label


# Apply preprocessing
train_data = (
    train_data
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .shuffle(10000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_data = (
    val_data
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_data = (
    test_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


# ============================================================
# 4. POSITIONAL EMBEDDING LAYER
# ============================================================

class PositionalEmbedding(layers.Layer):
    """
    Learns token embeddings + positional embeddings.
    """

    def __init__(self, max_len, vocab_size, embed_dim):
        super().__init__()

        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )

        self.position_embedding = layers.Embedding(
            input_dim=max_len,
            output_dim=embed_dim
        )

        self.max_len = max_len
        self.embed_dim = embed_dim

    def call(self, x):
        length = tf.shape(x)[-1]

        positions = tf.range(start=0, limit=length, delta=1)
        positions = self.position_embedding(positions)

        x = self.token_embedding(x)

        return x + positions


# ============================================================
# 5. TRANSFORMER ENCODER BLOCK
# ============================================================

class TransformerEncoder(layers.Layer):
    """
    Standard Transformer Encoder Block:
        - Multi-Head Self Attention
        - Feed Forward Network
        - Residual Connections
        - Layer Normalization
    """

    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])

        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs, training=False):

        # Self-attention
        attention_output = self.attention(
            inputs,
            inputs
        )

        attention_output = self.dropout1(
            attention_output,
            training=training
        )

        # Residual connection + normalization
        out1 = self.layernorm1(inputs + attention_output)

        # Feed-forward network
        ffn_output = self.ffn(out1)

        ffn_output = self.dropout2(
            ffn_output,
            training=training
        )

        # Residual connection + normalization
        return self.layernorm2(out1 + ffn_output)


# ============================================================
# 6. BUILD TRANSFORMER MODEL
# ============================================================

def build_transformer_model():
    """
    Creates a Transformer-based text classification model.
    """

    inputs = layers.Input(shape=(MAX_LEN,))

    # Token + positional embeddings
    x = PositionalEmbedding(
        MAX_LEN,
        VOCAB_SIZE,
        EMBED_DIM
    )(inputs)

    # Transformer Encoder
    x = TransformerEncoder(
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        ff_dim=FF_DIM
    )(x)

    # Global pooling
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dropout(0.2)(x)

    x = layers.Dense(64, activation="relu")(x)

    x = layers.Dropout(0.2)(x)

    # Binary classification output
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs)

    return model


model = build_transformer_model()

model.summary()


# ============================================================
# 7. COMPILE MODEL
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# ============================================================
# 8. TRAIN MODEL
# ============================================================

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        patience=2,
        restore_best_weights=True
    )
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=callbacks
)


# ============================================================
# 9. EVALUATE MODEL
# ============================================================

print("\nEvaluating on test dataset...")

test_loss, test_acc = model.evaluate(test_data)

print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")


# ============================================================
# 10. INFERENCE / PREDICTION
# ============================================================

sample_texts = [
    "This movie was absolutely fantastic. I loved it!",
    "Terrible film. Waste of time."
]

# Convert text -> token IDs
sample_tokens = vectorize_layer(tf.constant(sample_texts))

# Predict
predictions = model.predict(sample_tokens)

print("\nSample Predictions:\n")

for text, pred in zip(sample_texts, predictions):

    sentiment = "Positive" if pred[0] > 0.5 else "Negative"

    print(f"Text       : {text}")
    print(f"Prediction : {sentiment}")
    print(f"Confidence : {pred[0]:.4f}")
    print("-" * 50)

Loading IMDB dataset...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_embedding            │ (None, 200, 128)       │     2,585,600 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder             │ (None, 200, 128)       │       297,344 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,891,265 (11.03 MB)

 Trainable params: 2,891,265 (11.03 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 24s 20ms/step - accuracy: 0.6754 - loss: 0.5712 - val_accuracy: 0.8272 - val_loss: 0.3830
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.8680 - loss: 0.3155 - val_accuracy: 0.8682 - val_loss: 0.3136
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 23s 22ms/step - accuracy: 0.9167 - loss: 0.2193 - val_accuracy: 0.8736 - val_loss: 0.3217

Evaluating on test dataset...
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.8546 - loss: 0.3437

Test Loss     : 0.3437
Test Accuracy : 0.8546
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step

Sample Predictions:

Text       : This movie was absolutely fantastic. I loved it!
Prediction : Positive
Confidence : 0.8198
--------------------------------------------------
Text       : Terrible film. Waste of time.
Prediction : Negative
Confidence : 0.1043
--------------------------------------------------


```
How to adapt this code to other datasets:

1. Replace dataset loading section
   Examples:
       - AG News
       - Reuters
       - Custom CSV data
       - HuggingFace datasets

2. Change output layer:
       Multi-class:
           Dense(num_classes, activation='softmax')

3. Change loss:
       SparseCategoricalCrossentropy()

4. Tune:
       - EMBED_DIM
       - NUM_HEADS
       - FF_DIM
       - MAX_LEN

5. Add more TransformerEncoder blocks for deeper models.

Example:
    for _ in range(4):
        x = TransformerEncoder(...)(x)
```